# 06 Barcode Detection

**Purpose:** Fine-tune an existing OCT ResNet50 classifier into a binary classifier that detects the presence or absence of retinal barcoding.

**Inputs:**
- `data/processed/roi/`: preprocessed below-RPE ROI volumes from notebook 05.
- `data/processed/qc/preprocessing_qc.csv`: preprocessing status, ROI paths, scan dimensions, layer availability, and metadata.
- Existing pretrained ResNet50 checkpoint trained on the OCT-C8 dataset.
- `data/processed/labels/clinician/barcode_labels.csv`: clinician-provided barcode labels once available.

**Main task:**
Adapt the pretrained OCT-C8 ResNet50 into a binary barcode detector by replacing its original disease classification head with a binary classification head and fine-tuning it using clinician-provided barcode labels.

**Planned workflow:**
1. Load preprocessing QC table.
2. Load clinician labels.
3. Merge labels with ROI paths.
4. Convert volume-level labels and optional positive B-scan ranges into training examples.
5. Create patient-level train/validation/test splits to prevent patient leakage.
6. Load the pretrained OCT-C8 ResNet50 checkpoint.
7. Replace the original classification head with a binary barcode classification head.
8. Fine-tune the classifier, beginning with the final layers and optionally unfreezing deeper backbone layers.
9. Evaluate model performance on the held-out test set.
10. Generate Grad-CAM visualizations to verify that the model focuses on barcode regions rather than unrelated retinal structures.
11. Save trained models, predictions, and training history.

**Code organization:**
- Long reusable ResNet architecture, dataset loading, fine-tuning, inference, and evaluation code goes in `src/barcode/resnet.py`.
- Grad-CAM visualization code goes in `src/barcode/gradcam.py`.
- The notebook should remain lightweight, calling high-level functions, displaying summaries, inspecting outputs, and visualizing representative examples.

**Expected outputs:**
- `data/processed/models/barcode_resnet.pt`
- `data/processed/predictions/barcode_predictions.csv`
- `data/processed/qc/resnet_training_summary.csv`
- `data/processed/figures/gradcam/`

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
LABEL_FILE = PROCESSED_DIR / "labels" / "clinician" / "barcode_labels.csv"

C8_CHECKPOINT = PROJECT_ROOT / "results" / "models" / "resnet50_oct_c8_layer4_finetuned.pt"

In [ ]:
from barcode.resnet import (
    build_training_table,
    volume_to_slices_table,
    patient_split,
    run_resnet_finetuning,
)

In [ ]:
# Check whether clinician labels are available yet.

if LABEL_FILE.exists():
    print("Label file found:", LABEL_FILE)
else:
    print("Label file not found yet.")
    print("Expected location:", LABEL_FILE)

In [ ]:
# Preview training table once labels are available.

if LABEL_FILE.exists():
    volume_df = build_training_table(
        processed_dir=PROCESSED_DIR,
        label_file=LABEL_FILE,
        label_col="barcode_volume_status",
    )

    display(volume_df.head())
    print("Labeled volumes:", len(volume_df))
    print(volume_df["barcode_label"].value_counts())

In [ ]:
# Preview B-scan-level expansion once labels are available.

if LABEL_FILE.exists():
    slice_df = volume_to_slices_table(
        volume_df,
        use_positive_range=True,
    )

    display(slice_df.head())
    print("Training B-scans:", len(slice_df))
    print(slice_df["label"].value_counts())

In [ ]:
# Confirm patient-level split.

if LABEL_FILE.exists():
    train_df, val_df, test_df = patient_split(slice_df)

    print("Train patients:", train_df["patient_id"].nunique())
    print("Val patients:", val_df["patient_id"].nunique())
    print("Test patients:", test_df["patient_id"].nunique())

    print("Train rows:", len(train_df))
    print("Val rows:", len(val_df))
    print("Test rows:", len(test_df))

In [ ]:
# Fine-tune OCT-C8 ResNet50 into binary barcode detector.
# Run this only after labels are available.

if LABEL_FILE.exists():
    history_df, volume_pred_df = run_resnet_finetuning(
        processed_dir=PROCESSED_DIR,
        label_file=LABEL_FILE,
        c8_checkpoint_path=C8_CHECKPOINT,
        label_col="barcode_volume_status",
        use_positive_range=True,
        batch_size=16,
        epochs=5,
        lr=1e-4,
        weight_decay=1e-4,
        unfreeze_final_block=True,
    )

    display(history_df)
    display(volume_pred_df.head())

In [ ]:
# Confirm saved outputs.

for path in [
    PROCESSED_DIR / "models" / "barcode_resnet.pt",
    PROCESSED_DIR / "predictions" / "barcode_slice_predictions.csv",
    PROCESSED_DIR / "predictions" / "barcode_volume_predictions.csv",
    PROCESSED_DIR / "qc" / "resnet_training_summary.csv",
]:
    print(path, "exists:", path.exists())